# 02 — Workflow-Level Retrieval (L1 vs L0)

**Use case** : l'utilisateur sauvegarde des pipelines industriels nommés.  
Quand il décrit son intent, le système retrouve le bon workflow.

**Question** : est-ce que prédire un workflow (L1) est plus fiable que prédire des outils individuels (L0) ?

**Cible** : >80% R@1 en conditions réalistes.

### Données
- 7561 workflows n8n avec intent embeddings (1024D BGE-M3)
- Chaque workflow a un set de MCP tools mappés
- Stress test : 7561 workflows. Réaliste : 10-100 workflows utilisateur.

In [ ]:
import json
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

DATA_DIR = Path("../../gru/data")

# Load contrastive pairs (workflow intent embeddings + tool sets)
print("Loading n8n contrastive pairs...")
with open(DATA_DIR / "n8n-shgat-contrastive-pairs.json") as f:
    pairs = json.load(f)

print(f"  {len(pairs)} workflows")
print(f"  Sample: {pairs[0]['workflowName'][:60]}")
print(f"  Tools: {pairs[0]['positiveToolIds'][:4]}")

# Filter: keep only workflows with 2+ tools and valid embeddings
valid = [p for p in pairs if len(p['positiveToolIds']) >= 2 and len(p['intentEmbedding']) == 1024]
print(f"  Valid (2+ tools, 1024D): {len(valid)}")

# Extract arrays
wf_names = [p['workflowName'] for p in valid]
wf_ids = [p['workflowId'] for p in valid]
wf_embs = np.array([p['intentEmbedding'] for p in valid], dtype=np.float32)
wf_tools = [set(p['positiveToolIds']) for p in valid]

# Tool stats
all_tools = set()
for ts in wf_tools:
    all_tools.update(ts)
tool_lengths = [len(ts) for ts in wf_tools]
print(f"  Unique tools across all workflows: {len(all_tools)}")
print(f"  Tools/workflow: median={np.median(tool_lengths):.0f} avg={np.mean(tool_lengths):.1f} max={max(tool_lengths)}")

## Test 1 : Workflow Retrieval brut (cosine similarity)

Scénario : l'utilisateur décrit un intent → on cherche le workflow le plus proche par cosine.  
On simule en ajoutant du bruit à l'embedding d'un workflow connu (comme si l'utilisateur paraphrasait).

In [ ]:
np.random.seed(42)

def eval_workflow_retrieval(embs, n_queries=500, noise_std=0.1, subset_size=None):
    """
    Simulate intent queries as noised versions of workflow embeddings.
    If subset_size is given, randomly pick that many workflows as the catalog.
    """
    n_wf = len(embs)
    
    if subset_size and subset_size < n_wf:
        # Random subset (simulates user's personal catalog)
        idx = np.random.choice(n_wf, size=subset_size, replace=False)
        catalog_embs = embs[idx]
        catalog_map = {i: idx[i] for i in range(subset_size)}
    else:
        catalog_embs = embs
        idx = np.arange(n_wf)
        subset_size = n_wf
    
    # Generate queries from catalog workflows
    query_targets = np.random.choice(subset_size, size=n_queries)
    queries = catalog_embs[query_targets] + np.random.randn(n_queries, embs.shape[1]).astype(np.float32) * noise_std
    
    # Cosine similarity
    sims = cosine_similarity(queries, catalog_embs)
    
    ranks = []
    for i in range(n_queries):
        sorted_idx = np.argsort(-sims[i])
        rank = np.where(sorted_idx == query_targets[i])[0][0] + 1
        ranks.append(rank)
    ranks = np.array(ranks)
    
    r1 = (ranks == 1).mean() * 100
    r3 = (ranks <= 3).mean() * 100
    r5 = (ranks <= 5).mean() * 100
    mrr = (1.0 / ranks).mean()
    return {"r1": r1, "r3": r3, "r5": r5, "mrr": mrr}

print("=" * 75)
print("TEST 1 — Workflow retrieval par cosine (intent bruité → workflow)")
print("=" * 75)
print(f"{'Catalog':>8s}  {'Noise':>6s}  {'R@1':>7s}  {'R@3':>7s}  {'R@5':>7s}  {'MRR':>6s}")
print("-" * 55)

for catalog_size in [20, 50, 100, 500, None]:  # None = all 7561
    for noise in [0.05, 0.10, 0.20, 0.30]:
        label = str(catalog_size) if catalog_size else "ALL"
        r = eval_workflow_retrieval(wf_embs, n_queries=500, noise_std=noise, subset_size=catalog_size)
        flag = " ✓" if r['r1'] >= 80 else " ✗" if r['r1'] < 50 else ""
        print(f"{label:>8s}  {noise:>6.2f}  {r['r1']:>6.1f}%  {r['r3']:>6.1f}%  {r['r5']:>6.1f}%  {r['mrr']:>5.3f}{flag}")
    print()

## Test 2 : Cross-retrieval (un workflow retrouve-t-il ses voisins sémantiques ?)

Pas de bruit artificiel. On prend chaque workflow et on regarde si ses plus proches voisins
partagent les mêmes tools. C'est le test de cohérence de la hiérarchie.

In [ ]:
# For each workflow, find top-5 nearest neighbors and check tool overlap
print("=" * 75)
print("TEST 2 — Cohérence sémantique (voisins partagent-ils les mêmes tools ?)")
print("=" * 75)

# Sample 500 workflows to keep it fast
sample_idx = np.random.choice(len(valid), size=min(500, len(valid)), replace=False)
sample_embs = wf_embs[sample_idx]

sims = cosine_similarity(sample_embs, wf_embs)

overlaps_top1 = []
overlaps_top3 = []
overlaps_top5 = []

for i, si in enumerate(sample_idx):
    sorted_idx = np.argsort(-sims[i])
    # Skip self (will be rank 0)
    neighbors = [j for j in sorted_idx if j != si][:5]
    
    my_tools = wf_tools[si]
    for k, nb in enumerate(neighbors):
        nb_tools = wf_tools[nb]
        if len(my_tools) == 0:
            continue
        overlap = len(my_tools & nb_tools) / len(my_tools | nb_tools)  # Jaccard
        if k == 0:
            overlaps_top1.append(overlap)
        if k < 3:
            overlaps_top3.append(overlap)
        overlaps_top5.append(overlap)

print(f"  Jaccard tool overlap with nearest neighbor:  {np.mean(overlaps_top1):.3f} (avg)")
print(f"  Jaccard tool overlap with top-3 neighbors:   {np.mean(overlaps_top3):.3f} (avg)")
print(f"  Jaccard tool overlap with top-5 neighbors:   {np.mean(overlaps_top5):.3f} (avg)")
print(f"  Perfect match (Jaccard=1.0) in top-1:        {(np.array(overlaps_top1)==1.0).mean()*100:.1f}%")
print(f"  Some overlap (Jaccard>0) in top-1:           {(np.array(overlaps_top1)>0).mean()*100:.1f}%")

# Show examples: high overlap neighbors
print("\n  --- Exemples de voisins à fort overlap ---")
shown = 0
for i, si in enumerate(sample_idx):
    sorted_idx = np.argsort(-sims[i])
    nb = [j for j in sorted_idx if j != si][0]
    my_tools = wf_tools[si]
    nb_tools = wf_tools[nb]
    if len(my_tools | nb_tools) > 0:
        jacc = len(my_tools & nb_tools) / len(my_tools | nb_tools)
    else:
        continue
    if jacc > 0.5 and shown < 5:
        print(f"    sim={sims[i][nb]:.3f} J={jacc:.2f}")
        print(f"      A: {wf_names[si][:50]}  tools={list(my_tools)[:4]}")
        print(f"      B: {wf_names[nb][:50]}  tools={list(nb_tools)[:4]}")
        shown += 1

## Test 3 : L1 (workflow) vs L0 (tool) — quel niveau de prédiction est meilleur ?

Simulation : on a un intent et on veut trouver les bons tools.
- **L0 direct** : chercher les tools individuels par cosine (parmi tous les tools)
- **L1→L0** : chercher le bon workflow (L1), puis expand vers ses tools

Métrique : Tool Set F1 (combien de tools corrects dans le résultat)

In [ ]:
# Load tool embeddings for L0 comparison
print("Loading tool embeddings...")
with open(DATA_DIR / "n8n-node-embeddings.json") as f:
    tool_emb_data = json.load(f)

# Build tool embedding matrix
# tool_emb_data might be {toolId: embedding} or [{id, embedding}]
if isinstance(tool_emb_data, list):
    tool_emb_dict = {item['id']: item['embedding'] for item in tool_emb_data if 'embedding' in item}
elif isinstance(tool_emb_data, dict):
    tool_emb_dict = tool_emb_data
else:
    tool_emb_dict = {}

print(f"  Tool embeddings loaded: {len(tool_emb_dict)}")
if tool_emb_dict:
    sample_key = next(iter(tool_emb_dict))
    sample_val = tool_emb_dict[sample_key]
    if isinstance(sample_val, list):
        print(f"  Dim: {len(sample_val)} (sample: {sample_key})")
    elif isinstance(sample_val, dict) and 'embedding' in sample_val:
        print(f"  Nested dict format, extracting...")
        tool_emb_dict = {k: v['embedding'] for k, v in tool_emb_dict.items() if 'embedding' in v}
        print(f"  Extracted: {len(tool_emb_dict)} tools")

In [ ]:
# Build tool index from all tools appearing in workflows
all_tool_ids = sorted(all_tools)

# Find tools that have embeddings
tool_ids_with_emb = [t for t in all_tool_ids if t in tool_emb_dict]
print(f"Tools in workflows: {len(all_tool_ids)}")
print(f"Tools with embeddings: {len(tool_ids_with_emb)}")

if len(tool_ids_with_emb) == 0:
    # Maybe the mapping is n8n type -> embedding, not MCP tool ID
    print("\nTool embedding keys sample:", list(tool_emb_dict.keys())[:5])
    print("Workflow tool IDs sample:", list(all_tools)[:5])
    print("\n⚠️  No direct match — tool embeddings use different IDs than workflow tools")
    print("   This is expected: n8n node embeddings use n8n types, workflow tools use MCP IDs")
    print("   For L0 comparison, we'll use the workflow intent embeddings as proxy")

In [ ]:
# === L1 vs L0 comparison ===
# Since we may not have direct L0 tool embeddings, we simulate:
# - L1: retrieve the workflow → get its tool set → measure overlap
# - L0: for each query, find the most similar workflows and union their tools

np.random.seed(42)

def eval_l1_vs_l0(wf_embs, wf_tools, n_queries=500, noise=0.15):
    """Compare L1 (workflow retrieval) vs L0 (tool-level aggregation)."""
    n_wf = len(wf_embs)
    
    # Generate queries
    query_targets = np.random.choice(n_wf, size=n_queries)
    queries = wf_embs[query_targets] + np.random.randn(n_queries, wf_embs.shape[1]).astype(np.float32) * noise
    
    sims = cosine_similarity(queries, wf_embs)
    
    # L1: take top-1 workflow, use its tools
    l1_f1s = []
    l1_precisions = []
    l1_recalls = []
    
    # L0 aggregated: take top-k workflows, union their tools
    l0_agg_f1s = {k: [] for k in [1, 3, 5]}
    
    for i in range(n_queries):
        target = query_targets[i]
        true_tools = wf_tools[target]
        if len(true_tools) == 0:
            continue
        
        sorted_idx = np.argsort(-sims[i])
        
        # L1: top-1 workflow
        pred_tools = wf_tools[sorted_idx[0]]
        tp = len(true_tools & pred_tools)
        prec = tp / max(len(pred_tools), 1)
        rec = tp / len(true_tools)
        f1 = 2 * prec * rec / max(prec + rec, 1e-10)
        l1_f1s.append(f1)
        l1_precisions.append(prec)
        l1_recalls.append(rec)
        
        # L0 aggregated: union of top-k workflows' tools
        for k in [1, 3, 5]:
            union_tools = set()
            for j in range(k):
                union_tools.update(wf_tools[sorted_idx[j]])
            tp_k = len(true_tools & union_tools)
            prec_k = tp_k / max(len(union_tools), 1)
            rec_k = tp_k / len(true_tools)
            f1_k = 2 * prec_k * rec_k / max(prec_k + rec_k, 1e-10)
            l0_agg_f1s[k].append(f1_k)
    
    return {
        "l1_f1": np.mean(l1_f1s),
        "l1_precision": np.mean(l1_precisions),
        "l1_recall": np.mean(l1_recalls),
        "l1_exact": np.mean([f == 1.0 for f in l1_f1s]),
        "l0_agg_f1": {k: np.mean(v) for k, v in l0_agg_f1s.items()},
    }

print("=" * 75)
print("TEST 3 — L1 (workflow) vs L0 (tool aggregation) — Tool Set F1")
print("=" * 75)

for catalog_size in [20, 50, 100, 500, None]:
    if catalog_size:
        idx = np.random.choice(len(valid), size=catalog_size, replace=False)
        sub_embs = wf_embs[idx]
        sub_tools = [wf_tools[i] for i in idx]
    else:
        sub_embs = wf_embs
        sub_tools = wf_tools
        catalog_size = len(valid)
    
    r = eval_l1_vs_l0(sub_embs, sub_tools, noise=0.15)
    label = f"catalog={catalog_size}"
    print(f"\n  {label}")
    print(f"    L1 top-1 workflow:  F1={r['l1_f1']:.3f}  P={r['l1_precision']:.3f}  R={r['l1_recall']:.3f}  exact={r['l1_exact']*100:.1f}%")
    for k in [1, 3, 5]:
        print(f"    L0 top-{k} union:     F1={r['l0_agg_f1'][k]:.3f}")

## Test 4 : Clustering naturel — les workflows forment-ils des capabilities ?

Si des groupes de workflows partagent les mêmes tools, ils forment naturellement des L2+.  
On utilise k-means pour trouver des clusters et vérifier la cohérence.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("=" * 75)
print("TEST 4 — Clustering naturel des workflows")
print("=" * 75)

# Use a manageable subset
n_sample = min(2000, len(valid))
sample_idx = np.random.choice(len(valid), size=n_sample, replace=False)
sample_embs = wf_embs[sample_idx]
sample_tools = [wf_tools[i] for i in sample_idx]

for n_clusters in [10, 30, 50, 100, 200]:
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=3)
    labels = km.fit_predict(sample_embs)
    
    # Measure intra-cluster tool Jaccard
    intra_jaccards = []
    for c in range(n_clusters):
        members = np.where(labels == c)[0]
        if len(members) < 2:
            continue
        for ii in range(min(len(members), 10)):
            for jj in range(ii + 1, min(len(members), 10)):
                t1 = sample_tools[members[ii]]
                t2 = sample_tools[members[jj]]
                if len(t1 | t2) > 0:
                    intra_jaccards.append(len(t1 & t2) / len(t1 | t2))
    
    sil = silhouette_score(sample_embs, labels, sample_size=min(1000, n_sample))
    avg_jacc = np.mean(intra_jaccards) if intra_jaccards else 0
    high_jacc = np.mean([j > 0.5 for j in intra_jaccards]) if intra_jaccards else 0
    
    print(f"  k={n_clusters:>4d}  silhouette={sil:.3f}  intra-Jaccard={avg_jacc:.3f}  (>0.5: {high_jacc*100:.1f}%)")

## Test 5 : Simulation « App » — catalog personnel + paraphrases

Scénario réaliste :
1. L'utilisateur a 30 workflows sauvegardés
2. Il décrit ce qu'il veut avec des niveaux de bruit variés
3. Le système doit retrouver le bon workflow à >80%

On teste aussi : que se passe-t-il quand l'intent ne correspond à AUCUN workflow sauvegardé ?

In [ ]:
import matplotlib.pyplot as plt

np.random.seed(42)

def simulate_app(wf_embs, catalog_size=30, n_queries=300, noise_levels=None, ood_ratio=0.2):
    """
    Simulate app usage:
    - User has `catalog_size` saved workflows
    - (1-ood_ratio) queries match a saved workflow (with noise)
    - ood_ratio queries are out-of-distribution (from other workflows)
    """
    if noise_levels is None:
        noise_levels = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    
    n_wf = len(wf_embs)
    # Pick catalog
    catalog_idx = np.random.choice(n_wf, size=catalog_size, replace=False)
    catalog_embs = wf_embs[catalog_idx]
    
    # OOD pool (workflows NOT in catalog)
    ood_pool = np.array([i for i in range(n_wf) if i not in set(catalog_idx)])
    
    n_ood = int(n_queries * ood_ratio)
    n_id = n_queries - n_ood
    
    results = {}
    for noise in noise_levels:
        # In-distribution queries
        id_targets = np.random.choice(catalog_size, size=n_id)
        id_queries = catalog_embs[id_targets] + np.random.randn(n_id, wf_embs.shape[1]).astype(np.float32) * noise
        
        # OOD queries (should be rejected or get low confidence)
        ood_src = np.random.choice(ood_pool, size=n_ood)
        ood_queries = wf_embs[ood_src] + np.random.randn(n_ood, wf_embs.shape[1]).astype(np.float32) * noise
        
        # Eval in-distribution
        id_sims = cosine_similarity(id_queries, catalog_embs)
        id_ranks = []
        id_top_sims = []
        for i in range(n_id):
            sorted_idx = np.argsort(-id_sims[i])
            rank = np.where(sorted_idx == id_targets[i])[0][0] + 1
            id_ranks.append(rank)
            id_top_sims.append(id_sims[i][sorted_idx[0]])
        id_ranks = np.array(id_ranks)
        
        # Eval OOD (no correct answer exists)
        ood_sims = cosine_similarity(ood_queries, catalog_embs)
        ood_top_sims = [np.max(ood_sims[i]) for i in range(n_ood)]
        
        results[noise] = {
            "r1": (id_ranks == 1).mean() * 100,
            "r3": (id_ranks <= 3).mean() * 100,
            "mrr": (1.0 / id_ranks).mean(),
            "id_top_sim_avg": np.mean(id_top_sims),
            "ood_top_sim_avg": np.mean(ood_top_sims),
            "sim_gap": np.mean(id_top_sims) - np.mean(ood_top_sims),
        }
    
    return results

print("=" * 75)
print("TEST 5 — Simulation App (30 workflows, 20% OOD)")
print("=" * 75)

for cat_size in [10, 30, 50, 100]:
    print(f"\n  --- Catalog: {cat_size} workflows ---")
    app = simulate_app(wf_embs, catalog_size=cat_size)
    print(f"  {'Noise':>6s}  {'R@1':>7s}  {'R@3':>7s}  {'MRR':>6s}  {'ID sim':>7s}  {'OOD sim':>8s}  {'Gap':>5s}")
    for noise, r in app.items():
        flag = " ✓" if r['r1'] >= 80 else ""
        print(f"  {noise:>6.2f}  {r['r1']:>6.1f}%  {r['r3']:>6.1f}%  {r['mrr']:>5.3f}  {r['id_top_sim_avg']:>7.3f}  {r['ood_top_sim_avg']:>8.3f}  {r['sim_gap']:>5.3f}{flag}")

In [ ]:
# === Summary plot ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: R@1 vs noise for different catalog sizes
for cat_size in [10, 30, 50, 100]:
    app = simulate_app(wf_embs, catalog_size=cat_size)
    noises = sorted(app.keys())
    r1s = [app[n]['r1'] for n in noises]
    axes[0].plot(noises, r1s, 'o-', label=f'{cat_size} workflows')

axes[0].axhline(y=80, color='green', linestyle='--', alpha=0.5, label='Target 80%')
axes[0].set_xlabel('Query noise (σ)')
axes[0].set_ylabel('R@1 (%)')
axes[0].set_title('Workflow Retrieval R@1')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 105)
axes[0].grid(True, alpha=0.3)

# Plot 2: ID vs OOD similarity gap (rejection capability)
app30 = simulate_app(wf_embs, catalog_size=30)
noises = sorted(app30.keys())
id_sims = [app30[n]['id_top_sim_avg'] for n in noises]
ood_sims = [app30[n]['ood_top_sim_avg'] for n in noises]
axes[1].plot(noises, id_sims, 'o-', color='green', label='In-distribution')
axes[1].plot(noises, ood_sims, 'o-', color='red', label='Out-of-distribution')
axes[1].fill_between(noises, id_sims, ood_sims, alpha=0.15, color='blue')
axes[1].set_xlabel('Query noise (σ)')
axes[1].set_ylabel('Top-1 cosine similarity')
axes[1].set_title('ID vs OOD — Rejection Gap (30 wf)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Scaling — R@1 vs catalog size at fixed noise
cat_sizes = [5, 10, 20, 30, 50, 100, 200, 500, 1000, 2000]
r1_by_size = []
for cs in cat_sizes:
    if cs > len(valid):
        break
    r = eval_workflow_retrieval(wf_embs, n_queries=300, noise_std=0.15, subset_size=cs)
    r1_by_size.append(r['r1'])
axes[2].plot(cat_sizes[:len(r1_by_size)], r1_by_size, 'o-', color='#4c72b0')
axes[2].axhline(y=80, color='green', linestyle='--', alpha=0.5, label='Target 80%')
axes[2].set_xlabel('Catalog size')
axes[2].set_ylabel('R@1 (%)')
axes[2].set_title('R@1 vs Catalog Size (noise=0.15)')
axes[2].set_xscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('workflow-retrieval-results.png', dpi=120)
plt.show()
print("Saved: workflow-retrieval-results.png")

## Conclusions provisoires (tests 1-5)

- Avec un catalog de 10-50 workflows et bruit modéré : >80% R@1
- Le scaling dégrade naturellement (plus de candidats = plus d'ambiguïté)
- Le gap ID/OOD est faible → rejet des intents inconnus = difficile

**Mais** : tous ces tests supposent que l'intent match UN seul workflow.
Les tests 6-7 ci-dessous traitent le cas réel : intent multi-étapes → décomposition.

## Test 6 : Intent composite multi-étapes (le vrai problème)

L'utilisateur décrit quelque chose qui ne correspond à AUCUN workflow existant.
Son intent couvre 2-3 workflows qu'il faudrait combiner.

Simulation : on prend 2-3 workflows, on fusionne leurs embeddings (moyenne pondérée),
et on teste si le système peut retrouver TOUS les workflows sources.

C'est ici que le cosine top-1 échoue et qu'on a besoin de décomposition.

In [ ]:
np.random.seed(42)

def eval_composite_retrieval(wf_embs, wf_tools, wf_names, catalog_size=50,
                              n_queries=300, n_compose=2, noise=0.10):
    """
    Simulate composite intents that combine n_compose workflows.
    
    Composite intent = mean(wf_embs[sources]) + noise.
    Success = retrieving ALL source workflows in top-k.
    Tool recall = |retrieved_tools ∩ true_tools| / |true_tools|.
    """
    n_wf = len(wf_embs)
    
    # Pick catalog
    catalog_idx = np.random.choice(n_wf, size=min(catalog_size, n_wf), replace=False)
    catalog_embs = wf_embs[catalog_idx]
    catalog_tools_list = [wf_tools[i] for i in catalog_idx]
    cat_size = len(catalog_idx)
    
    results = {"top1": [], "top3": [], "top5": [], "topN": [],
               "tool_recall_1": [], "tool_recall_3": [], "tool_recall_5": [],
               "tool_recall_N": [], "tool_precision_N": [],
               "examples": []}
    
    for q in range(n_queries):
        # Pick n_compose random workflows from catalog
        src_local = np.random.choice(cat_size, size=n_compose, replace=False)
        
        # Composite intent = mean of source embeddings + noise
        composite = catalog_embs[src_local].mean(axis=0)
        composite += np.random.randn(wf_embs.shape[1]).astype(np.float32) * noise
        
        # True tool set = union of all source workflows' tools
        true_tools = set()
        for s in src_local:
            true_tools.update(catalog_tools_list[s])
        
        # Cosine similarity against catalog
        sims = cosine_similarity(composite.reshape(1, -1), catalog_embs)[0]
        sorted_idx = np.argsort(-sims)
        
        # Did we find ALL source workflows?
        src_set = set(src_local.tolist())
        for k_label, k_val in [("top1", 1), ("top3", 3), ("top5", 5), ("topN", n_compose)]:
            retrieved = set(sorted_idx[:k_val].tolist())
            found_all = src_set.issubset(retrieved)
            results[k_label].append(found_all)
        
        # Tool recall at different k
        for k_label, k_val in [("tool_recall_1", 1), ("tool_recall_3", 3),
                                ("tool_recall_5", 5), ("tool_recall_N", n_compose)]:
            retrieved_tools = set()
            for j in sorted_idx[:k_val]:
                retrieved_tools.update(catalog_tools_list[j])
            recall = len(true_tools & retrieved_tools) / max(len(true_tools), 1)
            results[k_label].append(recall)
        
        # Precision at k=n_compose
        retrieved_tools_N = set()
        for j in sorted_idx[:n_compose]:
            retrieved_tools_N.update(catalog_tools_list[j])
        precision_N = len(true_tools & retrieved_tools_N) / max(len(retrieved_tools_N), 1)
        results["tool_precision_N"].append(precision_N)
        
        # Save a few examples
        if q < 5:
            src_names = [wf_names[catalog_idx[s]] for s in src_local]
            ret_names = [wf_names[catalog_idx[sorted_idx[j]]] for j in range(min(5, cat_size))]
            results["examples"].append({
                "sources": src_names,
                "retrieved": ret_names,
                "true_tools": len(true_tools),
                "sim_top1": float(sims[sorted_idx[0]]),
            })
    
    return {k: np.mean(v) if isinstance(v[0], (bool, float, np.floating)) else v
            for k, v in results.items()}


print("=" * 75)
print("TEST 6 — Intent composite (2-3 workflows fusionnés)")
print("=" * 75)

for n_compose in [2, 3]:
    print(f"\n  === {n_compose} workflows combinés ===")
    for cat_size in [30, 50, 100]:
        r = eval_composite_retrieval(wf_embs, wf_tools, wf_names,
                                      catalog_size=cat_size,
                                      n_compose=n_compose, noise=0.10)
        print(f"\n  catalog={cat_size}:")
        print(f"    All sources found:  top-1={r['top1']*100:.1f}%"
              f"  top-{n_compose}={r['topN']*100:.1f}%"
              f"  top-3={r['top3']*100:.1f}%  top-5={r['top5']*100:.1f}%")
        print(f"    Tool recall:        @1={r['tool_recall_1']*100:.1f}%"
              f"  @{n_compose}={r['tool_recall_N']*100:.1f}%"
              f"  @3={r['tool_recall_3']*100:.1f}%  @5={r['tool_recall_5']*100:.1f}%")
        print(f"    Tool precision @{n_compose}: {r['tool_precision_N']*100:.1f}%")

    # Show examples for catalog=30
    r30 = eval_composite_retrieval(wf_embs, wf_tools, wf_names,
                                    catalog_size=30, n_compose=n_compose, noise=0.10)
    print(f"\n  --- Exemples (catalog=30, {n_compose} composés) ---")
    for ex in r30["examples"][:3]:
        print(f"    Sources: {[s[:40] for s in ex['sources']]}")
        print(f"    Top-3 retrieved: {[r[:40] for r in ex['retrieved'][:3]]}")
        print(f"    True tools: {ex['true_tools']}, top-1 sim: {ex['sim_top1']:.3f}")
        print()

## Test 7 : Greedy Set Cover — décomposition multi-workflow

Quand top-k cosine ne suffit pas, on peut utiliser une approche **Set Cover itérative** :
à chaque step, choisir le workflow qui couvre le PLUS de tools non encore couverts,
pondéré par la similarité cosine à l'intent.

C'est l'analogue offline de ce que le GRU ferait en séquentiel :
- Step 1 : quel workflow couvre le mieux l'intent ?
- Step 2 : quels tools restent non couverts ? quel workflow les couvre ?
- Repeat jusqu'à couverture complète ou score trop bas.

In [ ]:
def greedy_set_cover(query_emb, catalog_embs, catalog_tools, max_steps=5, sim_threshold=0.05):
    """
    Greedy set cover: iteratively pick the workflow that maximizes
    (cosine_sim * new_tools_covered) until no improvement.
    
    Returns list of (workflow_idx, sim, new_tools_covered).
    """
    sims = cosine_similarity(query_emb.reshape(1, -1), catalog_embs)[0]
    
    covered = set()
    selected = []
    used = set()
    
    for step in range(max_steps):
        best_score = -1
        best_idx = -1
        best_new = set()
        
        for j in range(len(catalog_embs)):
            if j in used:
                continue
            new_tools = catalog_tools[j] - covered
            if len(new_tools) == 0:
                continue
            # Score = cosine similarity * number of new tools covered
            score = sims[j] * len(new_tools)
            if score > best_score:
                best_score = score
                best_idx = j
                best_new = new_tools
        
        if best_idx == -1 or sims[best_idx] < sim_threshold:
            break
        
        selected.append((best_idx, float(sims[best_idx]), len(best_new)))
        covered.update(best_new)
        used.add(best_idx)
    
    return selected, covered


def eval_set_cover_vs_topk(wf_embs, wf_tools, wf_names, catalog_size=50,
                            n_queries=300, n_compose=2, noise=0.10):
    """Compare top-k cosine vs greedy set cover for composite intents."""
    n_wf = len(wf_embs)
    catalog_idx = np.random.choice(n_wf, size=min(catalog_size, n_wf), replace=False)
    catalog_embs = wf_embs[catalog_idx]
    catalog_tools_list = [wf_tools[i] for i in catalog_idx]
    cat_size = len(catalog_idx)
    
    topk_recalls = {1: [], 2: [], 3: [], 5: []}
    cover_recalls = []
    cover_precisions = []
    cover_steps = []
    topk_precisions = {1: [], 2: [], 3: [], 5: []}
    
    examples = []
    
    for q in range(n_queries):
        src_local = np.random.choice(cat_size, size=n_compose, replace=False)
        composite = catalog_embs[src_local].mean(axis=0)
        composite += np.random.randn(wf_embs.shape[1]).astype(np.float32) * noise
        
        true_tools = set()
        for s in src_local:
            true_tools.update(catalog_tools_list[s])
        
        # Top-k cosine
        sims = cosine_similarity(composite.reshape(1, -1), catalog_embs)[0]
        sorted_idx = np.argsort(-sims)
        for k in topk_recalls:
            ret_tools = set()
            for j in sorted_idx[:k]:
                ret_tools.update(catalog_tools_list[j])
            rec = len(true_tools & ret_tools) / max(len(true_tools), 1)
            prec = len(true_tools & ret_tools) / max(len(ret_tools), 1)
            topk_recalls[k].append(rec)
            topk_precisions[k].append(prec)
        
        # Greedy set cover
        selected, covered = greedy_set_cover(composite, catalog_embs, catalog_tools_list)
        rec = len(true_tools & covered) / max(len(true_tools), 1)
        prec = len(true_tools & covered) / max(len(covered), 1)
        cover_recalls.append(rec)
        cover_precisions.append(prec)
        cover_steps.append(len(selected))
        
        if q < 5:
            examples.append({
                "sources": [wf_names[catalog_idx[s]][:40] for s in src_local],
                "true_tools": len(true_tools),
                "topk_2_recall": topk_recalls[2][-1],
                "cover_recall": rec,
                "cover_steps": len(selected),
                "cover_selection": [(wf_names[catalog_idx[s[0]]][:35], s[1], s[2]) for s in selected[:4]],
            })
    
    return {
        "topk_recall": {k: np.mean(v) for k, v in topk_recalls.items()},
        "topk_precision": {k: np.mean(v) for k, v in topk_precisions.items()},
        "cover_recall": np.mean(cover_recalls),
        "cover_precision": np.mean(cover_precisions),
        "cover_steps_avg": np.mean(cover_steps),
        "examples": examples,
    }


print("=" * 75)
print("TEST 7 — Greedy Set Cover vs Top-k (intent = 2 workflows)")
print("=" * 75)

for cat_size in [30, 50, 100]:
    r = eval_set_cover_vs_topk(wf_embs, wf_tools, wf_names,
                                catalog_size=cat_size, n_compose=2, noise=0.10)
    print(f"\n  catalog={cat_size}:")
    print(f"    {'Method':<20s}  {'Recall':>8s}  {'Precision':>10s}")
    print(f"    {'-'*42}")
    for k in [1, 2, 3, 5]:
        print(f"    Top-{k} cosine       {r['topk_recall'][k]*100:>7.1f}%  {r['topk_precision'][k]*100:>9.1f}%")
    print(f"    Set Cover (avg {r['cover_steps_avg']:.1f}s)"
          f"  {r['cover_recall']*100:>7.1f}%  {r['cover_precision']*100:>9.1f}%")

# Examples
print("\n  --- Exemples détaillés (catalog=30) ---")
r30 = eval_set_cover_vs_topk(wf_embs, wf_tools, wf_names,
                              catalog_size=30, n_compose=2, noise=0.10)
for ex in r30["examples"][:3]:
    print(f"\n    Sources: {ex['sources']}")
    print(f"    True tools: {ex['true_tools']}  | Top-2 recall: {ex['topk_2_recall']*100:.0f}%  | Cover recall: {ex['cover_recall']*100:.0f}%")
    print(f"    Set cover steps ({ex['cover_steps']}):")
    for name, sim, new_t in ex["cover_selection"]:
        print(f"      → {name:<35s}  sim={sim:.3f}  +{new_t} tools")

# Same for 3-compose
print("\n" + "=" * 75)
print("TEST 7b — Greedy Set Cover vs Top-k (intent = 3 workflows)")
print("=" * 75)
for cat_size in [30, 50]:
    r = eval_set_cover_vs_topk(wf_embs, wf_tools, wf_names,
                                catalog_size=cat_size, n_compose=3, noise=0.10)
    print(f"\n  catalog={cat_size}:")
    print(f"    {'Method':<20s}  {'Recall':>8s}  {'Precision':>10s}")
    print(f"    {'-'*42}")
    for k in [1, 3, 5]:
        print(f"    Top-{k} cosine       {r['topk_recall'][k]*100:>7.1f}%  {r['topk_precision'][k]*100:>9.1f}%")
    print(f"    Set Cover (avg {r['cover_steps_avg']:.1f}s)"
          f"  {r['cover_recall']*100:>7.1f}%  {r['cover_precision']*100:>9.1f}%")